# Setup and Verify Task Manager Backend

This notebook installs dependencies, runs Django migrations, and verifies the backend API structure step by step.

In [1]:
import os
import subprocess
import sys

backend_dir = '/workspaces/task-manager/backend'
venv_dir = os.path.join(backend_dir, '.venv')

if not os.path.isdir(venv_dir):
    print('Creating virtual environment...')
    subprocess.run([sys.executable, '-m', 'venv', venv_dir], check=True)

pip_path = os.path.join(venv_dir, 'bin', 'pip')
print('Installing backend requirements...')
subprocess.run([pip_path, 'install', '--upgrade', 'pip'], check=True)
subprocess.run([pip_path, 'install', '-r', os.path.join(backend_dir, 'requirements.txt')], check=True)
print('Dependencies installed.')


Creating virtual environment...
The virtual environment was not created successfully because ensurepip is not
available.  On Debian/Ubuntu systems, you need to install the python3-venv
package using the following command.

    apt install python3.12-venv

You may need to use sudo with that command.  After installing the python3-venv
package, recreate your virtual environment.

Failing command: /workspaces/task-manager/backend/.venv/bin/python3.12



CalledProcessError: Command '['/workspaces/task-manager/.venv/bin/python', '-m', 'venv', '/workspaces/task-manager/backend/.venv']' returned non-zero exit status 1.

In [ ]:
import os
import subprocess

manage_py = os.path.join(backend_dir, 'manage.py')
python_path = os.path.join(venv_dir, 'bin', 'python')

print('Making migrations and applying them...')
subprocess.run([python_path, manage_py, 'makemigrations', 'api'], check=True)
subprocess.run([python_path, manage_py, 'migrate'], check=True)
print('Migrations applied successfully.')


In [ ]:
print('Running Django system checks...')
subprocess.run([python_path, manage_py, 'check'], check=True)
print('Django checks passed.')


In [2]:
import os
import subprocess
import sys

backend_dir = '/workspaces/task-manager/backend'
manage_py = os.path.join(backend_dir, 'manage.py')
python_path = sys.executable

print('Running Django makemigrations and migrate...')
subprocess.run([python_path, manage_py, 'makemigrations', 'api'], cwd=backend_dir, check=True)
subprocess.run([python_path, manage_py, 'migrate'], cwd=backend_dir, check=True)
print('Migrations complete.')

print('Running Django system checks...')
subprocess.run([python_path, manage_py, 'check'], cwd=backend_dir, check=True)
print('Django system checks passed.')


Running Django makemigrations and migrate...
No changes detected in app 'api'
Operations to perform:
  Apply all migrations: admin, api, auth, authtoken, contenttypes, sessions
Running migrations:
  Applying contenttypes.0001_initial... OK
  Applying auth.0001_initial... OK
  Applying admin.0001_initial... OK
  Applying admin.0002_logentry_remove_auto_add... OK
  Applying admin.0003_logentry_add_action_flag_choices... OK
  Applying api.0001_initial... OK
  Applying contenttypes.0002_remove_content_type_name... OK
  Applying auth.0002_alter_permission_name_max_length... OK
  Applying auth.0003_alter_user_email_max_length... OK
  Applying auth.0004_alter_user_username_opts... OK
  Applying auth.0005_alter_user_last_login_null... OK
  Applying auth.0006_require_contenttypes_0002... OK
  Applying auth.0007_alter_validators_add_error_messages... OK
  Applying auth.0008_alter_user_username_max_length... OK
  Applying auth.0009_alter_user_last_name_max_length... OK
  Applying auth.0010_alter_

In [3]:
import time
import urllib.request

print('Starting Django development server in background...')
server_process = subprocess.Popen([
    python_path,
    manage_py,
    'runserver',
    '0.0.0.0:8000'
], cwd=backend_dir, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

# Wait for the server to become ready
for i in range(10):
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/api/') as response:
            content = response.read(200)
            print('Server responded on /api/ with status', response.status)
            break
    except Exception as exc:
        print('Waiting for server...', exc)
        time.sleep(1)
else:
    server_process.terminate()
    raise RuntimeError('Server did not start in time')

print('Server is running.')
print('To stop it, you can terminate the process in the notebook or stop this kernel.')


Starting Django development server in background...
Waiting for server... <urlopen error [Errno 111] Connection refused>
Waiting for server... HTTP Error 401: Unauthorized
Waiting for server... HTTP Error 401: Unauthorized
Waiting for server... HTTP Error 401: Unauthorized
Waiting for server... HTTP Error 401: Unauthorized
Waiting for server... HTTP Error 401: Unauthorized
Waiting for server... HTTP Error 401: Unauthorized
Waiting for server... HTTP Error 401: Unauthorized
Waiting for server... HTTP Error 401: Unauthorized
Waiting for server... HTTP Error 401: Unauthorized


RuntimeError: Server did not start in time

In [4]:
import time
import urllib.request
import urllib.error

print('Starting Django development server in background...')
server_process = subprocess.Popen([
    python_path,
    manage_py,
    'runserver',
    '0.0.0.0:8000'
], cwd=backend_dir, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

ready = False
for i in range(12):
    if server_process.poll() is not None:
        stdout, stderr = server_process.communicate(timeout=1)
        raise RuntimeError(f'Server process exited early. stdout={stdout}\nstderr={stderr}')
    try:
        req = urllib.request.Request('http://127.0.0.1:8000/admin/login/')
        with urllib.request.urlopen(req, timeout=5) as response:
            if response.status in (200, 302):
                print('Server responded on /admin/login/ with status', response.status)
                ready = True
                break
    except urllib.error.HTTPError as exc:
        print('Server returned HTTP status', exc.code)
        if exc.code in (200, 302):
            ready = True
            break
    except Exception as exc:
        print('Waiting for server...', exc)
    time.sleep(1)

if not ready:
    server_process.terminate()
    raise RuntimeError('Server did not start in time')

print('Server is running.')
print('To stop it, terminate the notebook kernel or stop the server process.')


Starting Django development server in background...
Waiting for server... <urlopen error [Errno 111] Connection refused>
Server responded on /admin/login/ with status 200
Server is running.
To stop it, terminate the notebook kernel or stop the server process.


In [5]:
import http.client
import time
from pathlib import Path

frontend_dir = Path('/workspaces/task-manager/frontend')
print('Starting frontend HTTP server on port 3000...')
frontend_process = subprocess.Popen(
    [python_path, '-m', 'http.server', '3000'],
    cwd=frontend_dir,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)

for attempt in range(10):
    try:
        conn = http.client.HTTPConnection('127.0.0.1', 3000, timeout=5)
        conn.request('GET', '/')
        resp = conn.getresponse()
        body = resp.read(200).decode('utf-8', errors='ignore')
        if resp.status == 200 and '<title>Task Manager</title>' in body:
            print('Frontend server is running on http://127.0.0.1:3000')
            break
        print('Frontend started but page did not return expected content yet, status:', resp.status)
    except Exception as exc:
        print('Waiting for frontend server...', exc)
    time.sleep(1)
else:
    frontend_process.terminate()
    raise RuntimeError('Frontend server did not start in time')

print('Verifying backend API root...')
for attempt in range(5):
    try:
        conn = http.client.HTTPConnection('127.0.0.1', 8000, timeout=5)
        conn.request('GET', '/api/')
        resp = conn.getresponse()
        print('Backend API root status:', resp.status)
        break
    except Exception as exc:
        print('Waiting for backend API...', exc)
        time.sleep(1)
else:
    raise RuntimeError('Backend API did not respond')

print('Frontend and backend are both running.')
print('Stop servers by terminating the notebook kernel or the process objects.')


Starting frontend HTTP server on port 3000...
Waiting for frontend server... [Errno 111] Connection refused
Frontend server is running on http://127.0.0.1:3000
Verifying backend API root...
Backend API root status: 401
Frontend and backend are both running.
Stop servers by terminating the notebook kernel or the process objects.


In [6]:
import os

print('--- Setup Summary ---')
print('Backend directory:', backend_dir)
print('Virtualenv directory exists:', os.path.isdir(venv_dir))
print('Backend manage.py exists:', os.path.isfile(manage_py))
print('Backend server running:', server_process.poll() is None)
print('Frontend server running:', frontend_process.poll() is None)

try:
    conn = http.client.HTTPConnection('127.0.0.1', 8000, timeout=5)
    conn.request('GET', '/api/')
    resp = conn.getresponse()
    print('Backend /api/ status:', resp.status)
except Exception as exc:
    print('Backend /api/ error:', exc)

try:
    conn = http.client.HTTPConnection('127.0.0.1', 3000, timeout=5)
    conn.request('GET', '/')
    resp = conn.getresponse()
    print('Frontend / status:', resp.status)
except Exception as exc:
    print('Frontend / error:', exc)


--- Setup Summary ---
Backend directory: /workspaces/task-manager/backend
Virtualenv directory exists: True
Backend manage.py exists: True
Backend server running: True
Frontend server running: True
Backend /api/ status: 401
Frontend / status: 200


In [7]:
import socket
import subprocess
import http.client
import os

print('Checking server processes...')
for name, proc in [('backend', globals().get('server_process')), ('frontend', globals().get('frontend_process'))]:
    if proc is None:
        print(f'{name}: process object not found')
    else:
        print(f'{name}: alive={proc.poll() is None} returncode={proc.poll()}')

for host, port in [('127.0.0.1', 3000), ('127.0.0.1', 8000)]:
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.settimeout(1)
    try:
        sock.connect((host, port))
        print(f'{host}:{port} is reachable')
    except Exception as e:
        print(f'{host}:{port} is not reachable:', e)
    finally:
        sock.close()

for path in ['/', '/api/']:
    try:
        conn = http.client.HTTPConnection('127.0.0.1', 3000, timeout=2)
        conn.request('GET', path)
        resp = conn.getresponse()
        print('frontend', path, 'status', resp.status)
    except Exception as e:
        print('frontend', path, 'error', e)
    try:
        conn = http.client.HTTPConnection('127.0.0.1', 8000, timeout=2)
        conn.request('GET', path)
        resp = conn.getresponse()
        print('backend', path, 'status', resp.status)
    except Exception as e:
        print('backend', path, 'error', e)


Checking server processes...
backend: alive=True returncode=None
frontend: alive=True returncode=None
127.0.0.1:3000 is reachable
127.0.0.1:8000 is reachable
frontend / status 200
backend / status 404
frontend /api/ status 404
backend /api/ status 401


In [8]:
import time
import http.client
from pathlib import Path

# Stop previous frontend server if still running.
if 'frontend_process' in globals() and frontend_process is not None:
    if frontend_process.poll() is None:
        print('Stopping existing frontend process...')
        frontend_process.terminate()
        try:
            frontend_process.wait(timeout=5)
        except Exception:
            frontend_process.kill()
    else:
        print('Previous frontend process already stopped.')

frontend_dir = Path('/workspaces/task-manager/frontend')
print('Starting frontend HTTP server on port 5000...')
frontend_process = subprocess.Popen(
    [python_path, '-m', 'http.server', '5000'],
    cwd=frontend_dir,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)

for attempt in range(10):
    try:
        conn = http.client.HTTPConnection('127.0.0.1', 5000, timeout=5)
        conn.request('GET', '/')
        resp = conn.getresponse()
        body = resp.read(200).decode('utf-8', errors='ignore')
        if resp.status == 200 and '<title>Task Manager</title>' in body:
            print('Frontend server is running on http://127.0.0.1:5000')
            break
        print('Frontend responded with status:', resp.status)
    except Exception as exc:
        print('Waiting for frontend server...', exc)
        time.sleep(1)
else:
    frontend_process.terminate()
    raise RuntimeError('Frontend server did not start in time on port 5000')

print('Backend API root check:')
try:
    conn = http.client.HTTPConnection('127.0.0.1', 8000, timeout=5)
    conn.request('GET', '/api/')
    resp = conn.getresponse()
    print('backend /api/ status:', resp.status)
except Exception as exc:
    print('backend /api/ error:', exc)

print('Now use the forwarded port 5000 URL from your Codespaces Ports panel.')


Stopping existing frontend process...
Starting frontend HTTP server on port 5000...
Waiting for frontend server... [Errno 111] Connection refused
Frontend server is running on http://127.0.0.1:5000
Backend API root check:
backend /api/ status: 401
Now use the forwarded port 5000 URL from your Codespaces Ports panel.


In [9]:
import http.client
import time
from pathlib import Path

# Stop any existing frontend process.
if 'frontend_process' in globals() and frontend_process is not None:
    if frontend_process.poll() is None:
        frontend_process.terminate()
        try:
            frontend_process.wait(timeout=5)
        except Exception:
            frontend_process.kill()

frontend_dir = Path('/workspaces/task-manager/frontend')
frontend_port = 33351
print(f'Starting frontend HTTP server on port {frontend_port}...')
frontend_process = subprocess.Popen(
    [python_path, '-m', 'http.server', str(frontend_port)],
    cwd=frontend_dir,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)

for attempt in range(10):
    try:
        conn = http.client.HTTPConnection('127.0.0.1', frontend_port, timeout=5)
        conn.request('GET', '/')
        resp = conn.getresponse()
        body = resp.read(200).decode('utf-8', errors='ignore')
        if resp.status == 200 and '<title>Task Manager</title>' in body:
            print('Frontend server is running on http://127.0.0.1:' + str(frontend_port))
            break
        print('Frontend responded with status:', resp.status)
    except Exception as exc:
        print('Waiting for frontend server...', exc)
        time.sleep(1)
else:
    frontend_process.terminate()
    raise RuntimeError(f'Frontend server did not start in time on port {frontend_port}')

print('If port 33351 is forwarded in the Codespaces Ports panel, open that forwarded URL.')


Starting frontend HTTP server on port 33351...
Waiting for frontend server... timed out
Waiting for frontend server... timed out
Waiting for frontend server... timed out
Waiting for frontend server... timed out
Waiting for frontend server... timed out
Waiting for frontend server... timed out
Waiting for frontend server... timed out
Waiting for frontend server... timed out
Waiting for frontend server... timed out
Waiting for frontend server... timed out


RuntimeError: Frontend server did not start in time on port 33351

In [10]:
import time

print('frontend_process exists:', 'frontend_process' in globals())
if 'frontend_process' in globals():
    print('frontend_process poll:', frontend_process.poll())
    try:
        err = frontend_process.stderr.read()
        print('frontend_process stderr:', err)
    except Exception as e:
        print('stderr read error:', e)


frontend_process exists: True
frontend_process poll: 1
frontend_process stderr: Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/lib/python3.12/http/server.py", line 1314, in <module>
    test(
  File "/usr/lib/python3.12/http/server.py", line 1261, in test
    with ServerClass(addr, HandlerClass) as httpd:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socketserver.py", line 457, in __init__
    self.server_bind()
  File "/usr/lib/python3.12/http/server.py", line 1308, in server_bind
    return super().server_bind()
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/http/server.py", line 136, in server_bind
    socketserver.TCPServer.server_bind(self)
  File "/usr/lib/python3.12/socketserver.py", line 473, in server_bind
    self.socket.bind(self.server_address)
OSError: [Errno 98] Address already in use



In [11]:
import socket
import time
from pathlib import Path

# Terminate any existing frontend process.
if 'frontend_process' in globals() and frontend_process is not None:
    if frontend_process.poll() is None:
        print('Stopping existing frontend process...')
        frontend_process.terminate()
        try:
            frontend_process.wait(timeout=5)
        except Exception:
            frontend_process.kill()
    else:
        print('Existing frontend process already stopped.')

frontend_dir = Path('/workspaces/task-manager/frontend')
frontend_port = 33351
print(f'Checking port {frontend_port} availability...')
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    result = s.connect_ex(('127.0.0.1', frontend_port))
    print('connect_ex result for port', frontend_port, ':', result)

if result == 0:
    raise RuntimeError(f'Port {frontend_port} is already in use and cannot be started.')

print(f'Starting frontend HTTP server on port {frontend_port}...')
frontend_process = subprocess.Popen(
    [python_path, '-m', 'http.server', str(frontend_port)],
    cwd=frontend_dir,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)

for attempt in range(10):
    try:
        conn = http.client.HTTPConnection('127.0.0.1', frontend_port, timeout=5)
        conn.request('GET', '/')
        resp = conn.getresponse()
        body = resp.read(200).decode('utf-8', errors='ignore')
        if resp.status == 200 and '<title>Task Manager</title>' in body:
            print('Frontend server is running on http://127.0.0.1:' + str(frontend_port))
            break
        print('Frontend responded with status:', resp.status)
    except Exception as exc:
        print('Waiting for frontend server...', exc)
        time.sleep(1)
else:
    frontend_process.terminate()
    raise RuntimeError(f'Frontend server did not start in time on port {frontend_port}')

print('Now open the forwarded Codespaces URL for port', frontend_port)
print('Backend remains on port 8000')


Existing frontend process already stopped.
Checking port 33351 availability...
connect_ex result for port 33351 : 0


RuntimeError: Port 33351 is already in use and cannot be started.

In [12]:
import subprocess

print('Checking which process owns port 33351...')
try:
    result = subprocess.run(['lsof', '-i', ':33351', '-t'], capture_output=True, text=True, check=False)
    print('lsof stdout:', result.stdout.strip())
    print('lsof stderr:', result.stderr.strip())
    if result.stdout.strip():
        proc_id = int(result.stdout.strip().splitlines()[0])
        print('Found process id on 33351:', proc_id)
        subprocess.run(['kill', str(proc_id)], check=False)
        print('Sent kill to', proc_id)
    else:
        print('No process ID found on 33351')
except FileNotFoundError:
    print('lsof not available in this environment')
except Exception as exc:
    print('Error checking port owner:', exc)


Checking which process owns port 33351...


: 

In [ ]:
# Restart the Django server process so new URL routing is applied.
import time

if 'server_process' in globals() and server_process is not None:
    if server_process.poll() is None:
        print('Stopping existing Django server...')
        server_process.terminate()
        try:
            server_process.wait(timeout=5)
        except Exception:
            server_process.kill()
    else:
        print('Existing Django server already stopped.')

print('Starting Django server on 0.0.0.0:8000...')
server_process = subprocess.Popen([
    python_path,
    manage_py,
    'runserver',
    '0.0.0.0:8000'
], cwd=backend_dir, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

# Wait and verify root now serves the frontend page
import urllib.request
for i in range(10):
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/') as r:
            print('Root responded with', r.status)
            txt = r.read(200).decode('utf-8', errors='ignore')
            if '<title>Task Manager</title>' in txt:
                print('Frontend successfully served at root.')
                break
    except Exception as e:
        print('Waiting for server root...', e)
        time.sleep(1)
else:
    print('Failed to verify root after restart; check server logs.')


In [ ]:
# Diagnostic: inspect server process
print('server_process in globals:', 'server_process' in globals())
if 'server_process' in globals():
    print('server_process.poll():', server_process.poll())
    try:
        out = server_process.stdout.read()
        print('server stdout (snippet):', out[:1000])
    except Exception as e:
        print('stdout read error:', e)
    try:
        err = server_process.stderr.read()
        print('server stderr (snippet):', err[:1000])
    except Exception as e:
        print('stderr read error:', e)
